# WikiStreams lab
Read Wikipedia edits from Kafka and compare the windows. All times are UTC.

In [1]:
import sys
sys.path.insert(0, '/home/jovyan/jobs')
from wikistream_consumer import create_spark, read_events, aggregations, start_queries
spark = create_spark()

In [2]:
events = read_events(spark)
events.printSchema()
frames = aggregations(events)

root
 |-- title: string (nullable = true)
 |-- user: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- type: string (nullable = true)
 |-- wiki: string (nullable = true)
 |-- server_name: string (nullable = true)
 |-- bot: boolean (nullable = true)
 |-- minor: boolean (nullable = true)
 |-- length: struct (nullable = true)
 |    |-- old: long (nullable = true)
 |    |-- new: long (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- edit_delta: long (nullable = true)
 |-- edit_size: long (nullable = true)



## Bot edits
The demo counts bot and human edits in one-hour windows.

In [3]:
frames['bot_edits_1h'].explain()

== Physical Plan ==
*(5) HashAggregate(keys=[window#102-T120000ms, bot#21], functions=[count(1)])
+- StateStoreSave [window#102-T120000ms, bot#21], state info [ checkpoint = <unknown>, runId = 1b1755b9-6b7a-4c21-9e70-1b7cc33c4b71, opId = 0, ver = 0, numPartitions = 2] stateStoreCkptIds = None, Append, -9223372036854775808, -9223372036854775808, 2
   +- *(4) HashAggregate(keys=[window#102-T120000ms, bot#21], functions=[merge_count(1)])
      +- StateStoreRestore [window#102-T120000ms, bot#21], state info [ checkpoint = <unknown>, runId = 1b1755b9-6b7a-4c21-9e70-1b7cc33c4b71, opId = 0, ver = 0, numPartitions = 2] stateStoreCkptIds = None, 2
         +- *(3) HashAggregate(keys=[window#102-T120000ms, bot#21], functions=[merge_count(1)])
            +- Exchange hashpartitioning(window#102-T120000ms, bot#21, 2), REQUIRED_BY_STATEFUL_OPERATOR, [plan_id=75]
               +- *(2) HashAggregate(keys=[window#102-T120000ms, bot#21], functions=[partial_count(1)])
                  +- *(2) Project 

## Tumbling windows
A one-minute window has no overlap. Each edit belongs to one window. We count edits and measure the size change in bytes.

In [4]:
frames['tumbling_edits_1m'].explain()

== Physical Plan ==
*(5) HashAggregate(keys=[window#109-T120000ms, wiki#19, bot#21], functions=[count(1), sum(edit_delta#25L), sum(edit_size#28L), avg(edit_size#28L)])
+- StateStoreSave [window#109-T120000ms, wiki#19, bot#21], state info [ checkpoint = <unknown>, runId = bf5c4d6e-d105-4f82-a466-29d293965fe7, opId = 0, ver = 0, numPartitions = 2] stateStoreCkptIds = None, Append, -9223372036854775808, -9223372036854775808, 2
   +- *(4) HashAggregate(keys=[window#109-T120000ms, wiki#19, bot#21], functions=[merge_count(1), merge_sum(edit_delta#25L), merge_sum(edit_size#28L), merge_avg(edit_size#28L)])
      +- StateStoreRestore [window#109-T120000ms, wiki#19, bot#21], state info [ checkpoint = <unknown>, runId = bf5c4d6e-d105-4f82-a466-29d293965fe7, opId = 0, ver = 0, numPartitions = 2] stateStoreCkptIds = None, 2
         +- *(3) HashAggregate(keys=[window#109-T120000ms, wiki#19, bot#21], functions=[merge_count(1), merge_sum(edit_delta#25L), merge_sum(edit_size#28L), merge_avg(edit_size#

## Overlapping windows
The window lasts two minutes and starts every 30 seconds. Each edit belongs to four windows. Counts from these windows should not be added together.

In [5]:
frames['sliding_edits_2m'].explain()

== Physical Plan ==
*(5) HashAggregate(keys=[window#127-T120000ms, wiki#19], functions=[count(1), sum(edit_size#28L), max(edit_size#28L)])
+- StateStoreSave [window#127-T120000ms, wiki#19], state info [ checkpoint = <unknown>, runId = fc48b29a-a04d-4f37-85ed-8b1480d5239c, opId = 0, ver = 0, numPartitions = 2] stateStoreCkptIds = None, Append, -9223372036854775808, -9223372036854775808, 2
   +- *(4) HashAggregate(keys=[window#127-T120000ms, wiki#19], functions=[merge_count(1), merge_sum(edit_size#28L), merge_max(edit_size#28L)])
      +- StateStoreRestore [window#127-T120000ms, wiki#19], state info [ checkpoint = <unknown>, runId = fc48b29a-a04d-4f37-85ed-8b1480d5239c, opId = 0, ver = 0, numPartitions = 2] stateStoreCkptIds = None, 2
         +- *(3) HashAggregate(keys=[window#127-T120000ms, wiki#19], functions=[merge_count(1), merge_sum(edit_size#28L), merge_max(edit_size#28L)])
            +- Exchange hashpartitioning(window#127-T120000ms, wiki#19, 2), REQUIRED_BY_STATEFUL_OPERATOR, [

## Large edits
This query keeps edits with a size change of at least 500 bytes and groups them by page. The size is the absolute difference between the old and new page lengths.

In [6]:
frames['large_page_edits'].explain()

== Physical Plan ==
*(5) HashAggregate(keys=[window#138-T120000ms, title#15], functions=[count(1), sum(edit_size#28L)])
+- StateStoreSave [window#138-T120000ms, title#15], state info [ checkpoint = <unknown>, runId = f9b5a1cf-c4c0-4be1-9ea9-45f483e11106, opId = 0, ver = 0, numPartitions = 2] stateStoreCkptIds = None, Append, -9223372036854775808, -9223372036854775808, 2
   +- *(4) HashAggregate(keys=[window#138-T120000ms, title#15], functions=[merge_count(1), merge_sum(edit_size#28L)])
      +- StateStoreRestore [window#138-T120000ms, title#15], state info [ checkpoint = <unknown>, runId = f9b5a1cf-c4c0-4be1-9ea9-45f483e11106, opId = 0, ver = 0, numPartitions = 2] stateStoreCkptIds = None, 2
         +- *(3) HashAggregate(keys=[window#138-T120000ms, title#15], functions=[merge_count(1), merge_sum(edit_size#28L)])
            +- Exchange hashpartitioning(window#138-T120000ms, title#15, 2), REQUIRED_BY_STATEFUL_OPERATOR, [plan_id=406]
               +- *(2) HashAggregate(keys=[window#138

## Start the queries
Watch the output with `docker logs -f wikistreams_notebook`. Update mode prints changed rows. A two-minute watermark limits old state and may drop very late events.

In [7]:
for query in spark.streams.active:
    query.stop()
queries = start_queries(frames)
[(query.name, query.isActive) for query in queries]

[('bot_edits_1h', True),
 ('tumbling_edits_1m', True),
 ('sliding_edits_2m', True),
 ('large_page_edits', True)]

In [8]:
for query in queries:
    query.processAllAvailable()
    print(query.name, query.isActive, query.exception())
    assert query.exception() is None


bot_edits_1h 0 None


tumbling_edits_1m 0 None


sliding_edits_2m 0 None


large_page_edits 0 None


## Producer filters
The default is French Wikipedia edits. Use `--server en.wikipedia.org` for English, `--title France` for one exact page title, or `--bot no` for human edits. These filters run before sending events to Kafka.

To stop later, run `for query in queries: query.stop()` in a new cell.